In [20]:
batch_size = 64
block_size = 256
learning_rate = 3e-4
n_embed = 384
n_layers = 6
n_heads = 6
dropout = 0.2

batch_size = 4
block_size = 8
n_embed = 16
n_layers = 2
n_heads = 2
head_size = n_embed // n_heads

In [2]:
import requests
response = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")
text = response.text

print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

Vocabulary size: 65
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [4]:
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for i,c in enumerate(chars)}
encode = lambda s: [stoi[_s] for _s in s]
decode = lambda i: "".join(itos[_i] for _i in i)
print(encode("test"))
print(decode(encode("test")))

[58, 43, 57, 58]
test


In [5]:
data = encode(text)
data

[18,
 47,
 56,
 57,
 58,
 1,
 15,
 47,
 58,
 47,
 64,
 43,
 52,
 10,
 0,
 14,
 43,
 44,
 53,
 56,
 43,
 1,
 61,
 43,
 1,
 54,
 56,
 53,
 41,
 43,
 43,
 42,
 1,
 39,
 52,
 63,
 1,
 44,
 59,
 56,
 58,
 46,
 43,
 56,
 6,
 1,
 46,
 43,
 39,
 56,
 1,
 51,
 43,
 1,
 57,
 54,
 43,
 39,
 49,
 8,
 0,
 0,
 13,
 50,
 50,
 10,
 0,
 31,
 54,
 43,
 39,
 49,
 6,
 1,
 57,
 54,
 43,
 39,
 49,
 8,
 0,
 0,
 18,
 47,
 56,
 57,
 58,
 1,
 15,
 47,
 58,
 47,
 64,
 43,
 52,
 10,
 0,
 37,
 53,
 59,
 1,
 39,
 56,
 43,
 1,
 39,
 50,
 50,
 1,
 56,
 43,
 57,
 53,
 50,
 60,
 43,
 42,
 1,
 56,
 39,
 58,
 46,
 43,
 56,
 1,
 58,
 53,
 1,
 42,
 47,
 43,
 1,
 58,
 46,
 39,
 52,
 1,
 58,
 53,
 1,
 44,
 39,
 51,
 47,
 57,
 46,
 12,
 0,
 0,
 13,
 50,
 50,
 10,
 0,
 30,
 43,
 57,
 53,
 50,
 60,
 43,
 42,
 8,
 1,
 56,
 43,
 57,
 53,
 50,
 60,
 43,
 42,
 8,
 0,
 0,
 18,
 47,
 56,
 57,
 58,
 1,
 15,
 47,
 58,
 47,
 64,
 43,
 52,
 10,
 0,
 18,
 47,
 56,
 57,
 58,
 6,
 1,
 63,
 53,
 59,
 1,
 49,
 52,
 53,
 61,
 1,
 15,
 39,
 47,

In [6]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [7]:
train_data[:block_size+1]

[18,
 47,
 56,
 57,
 58,
 1,
 15,
 47,
 58,
 47,
 64,
 43,
 52,
 10,
 0,
 14,
 43,
 44,
 53,
 56,
 43,
 1,
 61,
 43,
 1,
 54,
 56,
 53,
 41,
 43,
 43,
 42,
 1,
 39,
 52,
 63,
 1,
 44,
 59,
 56,
 58,
 46,
 43,
 56,
 6,
 1,
 46,
 43,
 39,
 56,
 1,
 51,
 43,
 1,
 57,
 54,
 43,
 39,
 49,
 8,
 0,
 0,
 13,
 50,
 50,
 10,
 0,
 31,
 54,
 43,
 39,
 49,
 6,
 1,
 57,
 54,
 43,
 39,
 49,
 8,
 0,
 0,
 18,
 47,
 56,
 57,
 58,
 1,
 15,
 47,
 58,
 47,
 64,
 43,
 52,
 10,
 0,
 37,
 53,
 59,
 1,
 39,
 56,
 43,
 1,
 39,
 50,
 50,
 1,
 56,
 43,
 57,
 53,
 50,
 60,
 43,
 42,
 1,
 56,
 39,
 58,
 46,
 43,
 56,
 1,
 58,
 53,
 1,
 42,
 47,
 43,
 1,
 58,
 46,
 39,
 52,
 1,
 58,
 53,
 1,
 44,
 39,
 51,
 47,
 57,
 46,
 12,
 0,
 0,
 13,
 50,
 50,
 10,
 0,
 30,
 43,
 57,
 53,
 50,
 60,
 43,
 42,
 8,
 1,
 56,
 43,
 57,
 53,
 50,
 60,
 43,
 42,
 8,
 0,
 0,
 18,
 47,
 56,
 57,
 58,
 1,
 15,
 47,
 58,
 47,
 64,
 43,
 52,
 10,
 0,
 18,
 47,
 56,
 57,
 58,
 6,
 1,
 63,
 53,
 59,
 1,
 49,
 52,
 53,
 61,
 1,
 15,
 39,
 47,

In [8]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is [18] the target: 47
when input is [18, 47] the target: 56
when input is [18, 47, 56] the target: 57
when input is [18, 47, 56, 57] the target: 58
when input is [18, 47, 56, 57, 58] the target: 1
when input is [18, 47, 56, 57, 58, 1] the target: 15
when input is [18, 47, 56, 57, 58, 1, 15] the target: 47
when input is [18, 47, 56, 57, 58, 1, 15, 47] the target: 58
when input is [18, 47, 56, 57, 58, 1, 15, 47, 58] the target: 47
when input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47] the target: 64
when input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64] the target: 43
when input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43] the target: 52
when input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52] the target: 10
when input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10] the target: 0
when input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0] the target: 14
when input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14] the ta

In [9]:
import torch

# Device detection: prefer MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon) device")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA device")
else:
    device = torch.device("cpu")
    print("Using CPU device")

torch.manual_seed(1337)
ix = torch.randint(len(data) - block_size, (batch_size,))
ix

Using MPS (Apple Silicon) device


tensor([ 213173,  989153,  193174,  874116,  231497,  760195,   71893,  938070,
         376266,  672062,  664764,  591480,  977193,  401266,  450648,  852280,
         495115,  176472, 1092875,  373495,  828607,  516138,  758062,  712326,
         738304,  634909,  166352,   49147,   95723,  554568, 1107735,  324303,
         621322,  643369,  449681,  518249,  701743,  769987,  309039,  795116,
         903978,  975656,  386199, 1025017,  980432,   62815,  396895, 1051286,
         447447,  666273,  846830,  558922,  243798, 1044296,  226164,  362227,
         944048,  859913,  987658,  215918,  553352,  585205,  312731,  204832])

In [10]:
import torch
torch.manual_seed(1337)

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.tensor(data[i:i+block_size]) for i in ix]).to(device)
    y = torch.stack([torch.tensor(data[i+1:i+block_size+1]) for i in ix]).to(device)
    return x,y

xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

print("----")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([64, 256])
tensor([[ 0, 26, 53,  ..., 56, 43, 47],
        [60, 43, 56,  ..., 56,  1, 41],
        [26, 21, 33,  ..., 26, 21, 13],
        ...,
        [ 5, 57,  1,  ...,  1, 35, 47],
        [56, 53, 53,  ..., 59, 50, 42],
        [42, 47, 56,  ..., 39, 56,  1]], device='mps:0')
targets:
torch.Size([64, 256])
tensor([[26, 53, 58,  ..., 43, 47, 45],
        [43, 56,  1,  ...,  1, 41, 53],
        [21, 33, 31,  ..., 21, 13, 10],
        ...,
        [57,  1, 52,  ..., 35, 47, 50],
        [53, 53, 58,  ..., 50, 42,  1],
        [47, 56, 43,  ..., 56,  1, 51]], device='mps:0')
----
when input is [0] the target: 26
when input is [0, 26] the target: 53
when input is [0, 26, 53] the target: 58
when input is [0, 26, 53, 58] the target: 1
when input is [0, 26, 53, 58, 1] the target: 19
when input is [0, 26, 53, 58, 1, 19] the target: 50
when input is [0, 26, 53, 58, 1, 19, 50] the target: 53
when input is [0, 26, 53, 58, 1, 19, 50, 53] the target: 59
when input is [0, 26, 5

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) # (B, T, C)
        q = self.query(x) # (B, T, C)
        wei = q @ k.transpose(-2, -1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x) # (B, T, C)
        out = wei @ v # (B, T, C) @ (B, C, T) -> (B, T, T)
        return out

In [77]:

head_size = 8 // n_heads
print(dict(
    n_layers=n_layers,
    n_heads=n_heads,
    n_embed=n_embed,
    block_size=block_size,
    head_size=head_size,
    vocab_size=vocab_size
))

token_embedding_table = nn.Embedding(vocab_size, n_embed)
position_embedding_table = nn.Embedding(block_size, n_embed)
key = nn.Linear(n_embed, head_size, bias=False)
value = nn.Linear(n_embed, head_size, bias=False)


C = n_embed # TODO: is this accurate?
T = block_size

x,y = get_batch("train")
x = x.cpu() # (B, T)
x_emb = token_embedding_table(x) # (B, T, C)
print("x.shape:" + str(x.shape))
print("x_emb.shape:" + str(x_emb.shape))
print("--------------------------------")
query = nn.Linear(n_embed, head_size, bias=False)
Q = query(x_emb) # (B, T, head_size)
print("query.weights.shape:" + str(query.weight.shape))
print("Q.shape:" + str(Q.shape))
print("--------------------------------")
K = key(x_emb) # (B, T, head_size)
# TODO: why transpose and why this order?
Kt = K.transpose(-2, -1) # (B, head_size, T)
print("key.weights.shape:" + str(key.weight.shape))
print("K.shape:" + str(K.shape))
print("Kt.shape:" + str(Kt.shape))
print("--------------------------------")
V = value(x_emb) # (B, T, head_size)
print("value.weights.shape:" + str(value.weight.shape))
print("V.shape:" + str(V.shape))
print("--------------------------------")

attention_weights = Q @ Kt# * C**-0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
print("attention_weights.shape:" + str(attention_weights.shape))
print("--------------------------------")

triangle = torch.tril(torch.ones(block_size, block_size))
print("triangle.shape:" + str(triangle.shape))
mask = triangle == 0 #[:T, :T] == 0
print("mask.shape:" + str(mask.shape))
attention_weights = attention_weights.masked_fill(mask, float("-inf"))
print("attention_weights.shape:" + str(attention_weights.shape))
attention_weights = F.softmax(attention_weights, dim=-1)
print("--------------------------------")



out = attention_weights @ V # (B, T, T) @ (B, T, C) -> (B, T, C)
print("out.shape:" + str(out.shape))

{'n_layers': 2, 'n_heads': 2, 'n_embed': 16, 'block_size': 8, 'head_size': 4, 'vocab_size': 65}
x.shape:torch.Size([4, 8])
x_emb.shape:torch.Size([4, 8, 16])
--------------------------------
query.weights.shape:torch.Size([4, 16])
Q.shape:torch.Size([4, 8, 4])
--------------------------------
key.weights.shape:torch.Size([4, 16])
K.shape:torch.Size([4, 8, 4])
Kt.shape:torch.Size([4, 4, 8])
--------------------------------
value.weights.shape:torch.Size([4, 16])
V.shape:torch.Size([4, 8, 4])
--------------------------------
attention_weights.shape:torch.Size([4, 8, 8])
--------------------------------
triangle.shape:torch.Size([8, 8])
mask.shape:torch.Size([8, 8])
attention_weights.shape:torch.Size([4, 8, 8])
--------------------------------
out.shape:torch.Size([4, 8, 4])


In [83]:
attention_weights

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4263, 0.5737, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4463, 0.3063, 0.2474, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3990, 0.1346, 0.2387, 0.2277, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1600, 0.1554, 0.4035, 0.0980, 0.1831, 0.0000, 0.0000, 0.0000],
         [0.1940, 0.1386, 0.1214, 0.1904, 0.1247, 0.2309, 0.0000, 0.0000],
         [0.1467, 0.2061, 0.2251, 0.1294, 0.1470, 0.1068, 0.0389, 0.0000],
         [0.0711, 0.0949, 0.1875, 0.0525, 0.1273, 0.0720, 0.0989, 0.2959]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3506, 0.6494, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2549, 0.5600, 0.1851, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1576, 0.3533, 0.2885, 0.2006, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2379, 0.1447, 0.1803, 0.2219, 0.2151, 0.0000, 0.0000, 0.0000],
         [0.1599, 0.351

In [ ]:
triangle = torch.tril(torch.ones(block_size, block_size))
mask = triangle[:block_size, :block_size] == 0
mask

tensor([[False,  True,  True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True],
        [False, False, False, False, False, False, False,  True],
        [False, False, False, False, False, False, False, False]])

In [65]:
Q @ K 

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [4, 4] but got: [4, 8].

In [47]:
print("x_emb.shape:" + str(x_emb.shape))
query = nn.Linear(n_embed, head_size, bias=False)
print("query.weights.shape:" + str(query.weight.shape))
query(x_emb).shape

x_emb.shape:torch.Size([4, 8, 16])
query.weights.shape:torch.Size([8, 16])


torch.Size([4, 8, 8])

In [48]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [49]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


In [50]:
class Block(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
        
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [51]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (B, T, C)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        #x = self.sa_heads(x)
        #x = self.ffwd(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    #@torch.inference_mode()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:,-1,:]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
m = m.to(device)  # Move model to device
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long).to(device)
#m.generate(1, 10)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

TypeError: MultiHeadAttention.__init__() takes 2 positional arguments but 3 were given

In [16]:
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [17]:
for steps in range(5000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    if steps % 100 == 0:
        print(f"step {steps} loss: {loss.item()}")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step 0 loss: 4.4974164962768555
step 100 loss: 2.50821852684021
step 200 loss: 2.4540915489196777
step 300 loss: 2.3988804817199707
step 400 loss: 2.3228139877319336
step 500 loss: 2.1654152870178223
step 600 loss: 2.0493171215057373
step 700 loss: 1.9461336135864258
step 800 loss: 1.847559928894043
step 900 loss: 1.7786173820495605
step 1000 loss: 1.7582967281341553
step 1100 loss: 1.7622077465057373
step 1200 loss: 1.6735448837280273
step 1300 loss: 1.6281712055206299
step 1400 loss: 1.5833464860916138
step 1500 loss: 1.5895500183105469
step 1600 loss: 1.550560712814331
step 1700 loss: 1.4915553331375122
step 1800 loss: 1.4904577732086182
step 1900 loss: 1.4784373044967651
step 2000 loss: 1.441312551498413
step 2100 loss: 1.4454877376556396
step 2200 loss: 1.4282934665679932
step 2300 loss: 1.4533597230911255
step 2400 loss: 1.3899226188659668
step 2500 loss: 1.3624910116195679
step 2600 loss: 1.397514820098877
step 2700 loss: 1.336158275604248
step 2800 loss: 1.3621097803115845
step

In [18]:
idx = torch.zeros((1, 1), dtype=torch.long).to(device)
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))


est one that.

AUTOLYCUS:
What, have you would?

Shepherd:
Done your highness, father, cry 'GainsaN, you deserveres,
you'ld Ratchard, you be well here brief and pray,
For that, you would pursuivantly, that would bent,
Yet seeing to revery mercy hold you on.

CAMILLO:
Behidden, then grandly know now
So sovereign of her two much, if thou do,
As well thou alto thy sicksand, at chold of doors,
When trift fiends that less, that once the frowns
Of word beaten that heaven a thousand be?
And when bid th
